# Unit 2 — A2C + GAE: From VPG to Actor-Critic

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

In Unit 1 we saw that **VPG with a value baseline** (Algorithm 1) still collapses
on harder environments — the culprit is the **Monte Carlo return** $R_t$, which
is computed from the entire future of the episode and is therefore very
high-variance.

This unit introduces two ideas that together form **A2C** (Advantage Actor-Critic):

1. **GAE** (Generalised Advantage Estimation) — replaces the MC return with a
   smooth interpolation between low-bias MC and low-variance 1-step TD.
2. **N-step rollouts** — instead of waiting for a full episode before updating,
   collect $N$ environment steps, bootstrap the return from $V(s_{t+N})$, and
   update immediately. This is the core of **A3C** (Mnih et al., 2016) and its
   synchronous variant A2C.

**What we build:**

| Part | Method | Key change |
|------|--------|-----------|
| A | VPG + value baseline (Unit 1 baseline) | Per-episode MC advantage |
| B | A2C + GAE | N-step rollouts · GAE(λ=0.95) · parallel envs |

Both parts use identical network architectures. The only differences are the
advantage estimator and the rollout collection strategy.


In [ ]:
# swig builds the box2d physics engine used by LunarLander
%pip install -q swig
%pip install -q "gymnasium[box2d]" imageio pillow matplotlib torch
print("All packages ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from IPython.display import Image, display

print(f"gymnasium {gym.__version__}  |  torch {torch.__version__}")


In [ ]:
# Note: for small networks like this one, CPU is often faster than GPU.
# GPU wins with large batch sizes and large models; here the overhead of
# moving tensors to the GPU outweighs the compute benefit.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## 1  The Environment — LunarLander-v3

```
              ___
             /   \
            | ◉   |   ← lander body
             \_|_/
               |
              /|\
             / | \    ← engine plumes
        ____/__|__\____
       |   landing pad  |
```

The lander starts at the top of the screen with a random initial velocity.
Gravity pulls it down. The agent fires thrusters to slow the descent and
steer toward the landing pad.

### State — 8 continuous values

| # | Symbol | Meaning |
|---|--------|---------|
| 0 | $x$ | Horizontal position |
| 1 | $y$ | Vertical position |
| 2 | $\dot{x}$ | Horizontal velocity |
| 3 | $\dot{y}$ | Vertical velocity |
| 4 | $\theta$ | Tilt angle |
| 5 | $\dot{\theta}$ | Angular velocity |
| 6 | $c_L$ | Left leg contact (0 or 1) |
| 7 | $c_R$ | Right leg contact (0 or 1) |

### Actions — 4 discrete

| Action | Effect |
|--------|--------|
| 0 | Do nothing |
| 1 | Fire left thruster |
| 2 | Fire main engine (brake) |
| 3 | Fire right thruster |

### Reward structure

The reward is **dense** — the agent gets useful signal every step:

$$r_t = -\text{(distance to pad)} - \text{speed} - 0.03\,|\theta| + 10\,c_L + 10\,c_R$$

Plus terminal bonuses: $+100$ for a safe landing, $-100$ for a crash.

### Why LunarLander is harder than CartPole

| | CartPole | LunarLander |
|--|----------|-------------|
| Episode length | up to 500 steps | up to 1000 steps |
| Actions to coordinate | 1 (left/right) | 3 thrusters |
| Credit assignment | Easy (reward = 1/step) | Hard (landing bonus at end) |
| Solved threshold | avg $\geq 475$ | avg $\geq 200$ |

This is why VPG — which worked on CartPole — fails here, and why we need
better variance reduction.


In [ ]:
env_peek = gym.make("LunarLander-v3")
STATE_DIM  = env_peek.observation_space.shape[0]   # 8
ACTION_DIM = env_peek.action_space.n               # 4
print(f"State dim : {STATE_DIM}")
print(f"Action dim: {ACTION_DIM}")
env_peek.close()


---
## 2  What's Wrong with VPG on LunarLander?

Recall VPG's advantage from Unit 1:

$$\hat{A}_t = R_t - V(s_t), \qquad R_t = \sum_{t'=t}^{T} \gamma^{t'-t} r_{t'}$$

$R_t$ is the **full Monte Carlo return** — it looks all the way to the end of the
episode.  On CartPole (max 500 steps, $+1$/step) this was noisy but manageable.
On LunarLander:

- Episodes can last 1000 steps, with a crash bonus only at step 1000
- A single bad landing wipes out all credit accumulated over 999 good steps
- $R_t$ for an early timestep averages over that entire uncertain future

**The variance of $R_t$ scales with episode length.**  On LunarLander it is simply
too noisy for the policy gradient to converge reliably.


---
## 3  The Bias-Variance Trade-off

There is a spectrum of ways to estimate the return $R_t$:

### k-step returns

$$R_t^{(k)} = r_t + \gamma r_{t+1} + \cdots + \gamma^{k-1} r_{t+k-1} + \gamma^k V(s_{t+k})$$

After $k$ real rewards, we **bootstrap** — we stop and use the value function
$V(s_{t+k})$ as a proxy for the remaining future.

| $k$ | Estimator | Variance | Bias |
|---|---|---|---|
| 1 | $r_t + \gamma V(s_{t+1})$ | **Low** | High (trusts $V$ completely) |
| 5 | 5-step return + bootstrap | Medium | Medium |
| $\infty$ | $R_t$ (full MC) | **High** | Zero |

### The TD residual $\delta_t$

Define the **one-step TD residual**:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is the immediate surprise: *"how much better was reality than predicted?"*

The $k$-step advantage can be written as a sum of TD residuals:

$$R_t^{(k)} - V(s_t) = \sum_{l=0}^{k-1} \gamma^l \delta_{t+l}$$


---
## 4  Generalised Advantage Estimation (Schulman et al., ICLR 2016)

Instead of choosing a single $k$, GAE takes an **exponentially-weighted
average over all $k$-step estimates**:

$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = (1-\lambda)\sum_{k=1}^{\infty}
  \lambda^{k-1}\bigl(R_t^{(k)} - V(s_t)\bigr)$$

Weights $(1-\lambda)\lambda^0,\;(1-\lambda)\lambda^1,\;\ldots$ sum to 1.

### The beautiful simplification

Substituting $R_t^{(k)} - V(s_t) = \sum_{l=0}^{k-1}\gamma^l\delta_{t+l}$ and
collecting terms, the infinite sum collapses to a single recurrence:

$$\boxed{\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{\infty}(\gamma\lambda)^l\,\delta_{t+l}}$$

### Two special cases

$$\lambda = 0:\quad \hat{A}_t = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
\qquad \text{(1-step TD — low variance, high bias)}$$

$$\lambda = 1:\quad \hat{A}_t = R_t - V(s_t)
\qquad \text{(Monte Carlo — low bias, high variance)}$$

$\lambda = 0.95$ is the standard choice: uses roughly $\frac{1}{1-0.95} = 20$
effective steps, enough to see the consequence of actions while keeping
variance manageable.

### O(T) backward recurrence

In practice we never compute the infinite sum explicitly.  At the end of a
rollout of length $T$ we walk backward:

$$\hat{A}_T = 0$$
$$\hat{A}_t = \delta_t + \gamma\lambda\,\hat{A}_{t+1}$$

Crucially: when step $t$ is a **terminal step** (episode ended), the next
state belongs to a new episode — we must *not* bootstrap across that boundary.
We multiply the $\hat{A}_{t+1}$ term by $(1 - \text{done}_t)$:

$$\hat{A}_t = \delta_t + \gamma\lambda\,(1 - \text{done}_t)\,\hat{A}_{t+1}$$


In [ ]:
def compute_gae(
    rewards:     torch.Tensor,   # (T, N_envs)  — rewards collected
    values:      torch.Tensor,   # (T, N_envs)  — V(s_t), DETACHED
    dones:       torch.Tensor,   # (T, N_envs)  — 1.0 if episode ended at step t
    next_values: torch.Tensor,   # (N_envs,)    — bootstrap V(s_{T+1})
    gamma: float = 0.99,
    lam:   float = 0.95,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute GAE advantages and critic targets for an N-step rollout.

    The key addition over the Unit-1 version: the (1 - done_t) mask
    prevents GAE from bleeding across episode boundaries.  When a
    LunarLander episode ends mid-rollout (crash or landing), the
    next step belongs to a fresh episode — bootstrapping across that
    boundary would inject the wrong value estimate.

    Parameters
    ----------
    rewards     : collected r_t  for each env in parallel
    values      : V(s_t) predictions from the critic, detached
    dones       : 1.0 on the step where the episode ended, else 0.0
    next_values : V(s_{T+1}) — bootstrap value after the last step
    gamma       : discount factor
    lam         : GAE lambda  (0 → 1-step TD,  1 → Monte Carlo)

    Returns
    -------
    advantages  : (T*N_envs,) — normalised GAE estimates, flattened
    returns     : (T*N_envs,) — critic regression targets R_t = Â_t + V(s_t)

    Example (T=3, N=1)
    ------------------
    t=2 (terminal): done=1  →  next_values masked to 0
        δ_2 = r_2 + γ·0         - V(s_2)
        Â_2 = δ_2
    t=1:
        δ_1 = r_1 + γ·V(s_2)   - V(s_1)
        Â_1 = δ_1 + γλ·(1-done_1)·Â_2  ← done_1=0, so Â_2 propagates
    t=0:
        δ_0 = r_0 + γ·V(s_1)   - V(s_0)
        Â_0 = δ_0 + γλ·(1-done_0)·Â_1
    """
    T, N = rewards.shape

    advantages = torch.zeros_like(rewards)   # (T, N_envs)
    gae        = torch.zeros(N, device=rewards.device)

    for t in reversed(range(T)):
        # non-terminal mask: if done at step t, don't propagate GAE forward
        nonterminal = 1.0 - dones[t]                              # (N_envs,)

        # next-step value: use bootstrap for the last step, else V(s_{t+1})
        next_v = values[t + 1] if t < T - 1 else next_values      # (N_envs,)

        # ── 1-step TD residual ────────────────────────────────────────────────
        delta = rewards[t] + gamma * next_v * nonterminal - values[t]
        # delta: (N_envs,)

        # ── GAE backward recurrence ───────────────────────────────────────────
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae

    # ── Critic targets: R_t = Â_t + V(s_t) ──────────────────────────────────
    returns = advantages + values                  # (T, N_envs)

    # ── Flatten and normalise advantages ─────────────────────────────────────
    adv_flat = advantages.reshape(-1)              # (T*N_envs,)
    ret_flat = returns.reshape(-1)                 # (T*N_envs,)

    adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)

    return adv_flat, ret_flat


---
## 5  A3C and A2C — N-step Rollouts with Parallel Workers

### A3C (Mnih et al., 2016)

A3C runs **multiple workers** in parallel, each with its own copy of the
environment.  Every worker:

1. Steps forward $N$ steps in its environment
2. Computes gradients using the $N$-step return + GAE
3. Asynchronously pushes those gradients to a **shared global network**

The asynchrony acts like experience replay: different workers are at
different points in their episodes, so each gradient batch is decorrelated
from the others.

### A2C — the synchronous version

**A2C** (the synchronous variant) waits for *all* workers to finish their
$N$-step rollout, concatenates the experiences, and does one synchronised
update.  In practice A2C is simpler to implement and equally effective on
single-machine training.

We implement A2C here: $N_e$ parallel environments stepping together.

### Why N-step rollouts fix the per-episode instability

| Per-episode VPG | N-step A2C |
|--|--|
| Wait for full episode (100–1000 steps) | Update every $N=5$ steps |
| Gradient norm ∝ episode length | Gradient norm is constant |
| One bad episode → catastrophic update | One bad episode → small contribution |
| No exploitation of parallelism | $N_e$ environments → $N_e\times$ data/sec |

The combination of **shorter rollouts + episode-boundary masking in GAE**
is what eliminates the collapse we saw with VPG in Unit 1.


---
## 6  Networks

### Orthogonal initialisation

Standard random initialisation (e.g. Kaiming) can produce policy logits
that are very close together, making the initial policy nearly uniform.
**Orthogonal initialisation** with small `std=0.01` on the actor's last
layer pushes the initial logits apart more slowly, producing a near-uniform
distribution *by design* and giving GAE a clean signal to work with from
the start.

### Logits vs probabilities

We feed **raw logits** (not softmax probabilities) to `Categorical`.
PyTorch computes `log_prob` directly from logits via the log-sum-exp trick,
which is more numerically stable than computing `softmax → log(p)`.

### Separate actor and critic

We keep the actor and critic as **separate networks** (no shared trunk).
This makes the learning dynamics cleaner to reason about and matches what
most reference implementations use for discrete-action environments.


In [ ]:
def layer_init(layer: nn.Linear, std: float = np.sqrt(2)) -> nn.Linear:
    """
    Orthogonal initialisation — standard for A2C/PPO (see CleanRL, SB3).

    Orthogonal matrices preserve vector norms: ‖Wx‖ = ‖x‖.
    This prevents gradient vanishing/exploding at initialisation.
    The std argument scales the matrix:
      std=√2  for hidden layers (accounts for Tanh saturation)
      std=1   for the critic output  (value predictions start near 0)
      std=0.01 for the actor output  (near-uniform initial policy)
    """
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, 0.0)
    return layer


class ActorNetwork(nn.Module):
    """
    Maps state → action logits (NOT probabilities).
    Input:  (state_dim,)    e.g. (8,) for LunarLander
    Output: (action_dim,)   raw logits, passed to Categorical(logits=...)
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),   # (8,)   → (256,)
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),       # (256,) → (256,)
            nn.Tanh(),
            layer_init(nn.Linear(hidden, action_dim), std=0.01),  # → (4,) logits
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)   # raw logits — (action_dim,) or (..., action_dim)

    def get_action(self, state: torch.Tensor, action: torch.Tensor = None):
        """
        Sample (or evaluate) an action and return log_prob + entropy.

        action=None  →  sample from the distribution
        action=given →  evaluate log_prob of that action (used during the
                         update step when we re-evaluate stored actions)
        """
        logits = self.forward(state)
        dist   = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy()


class CriticNetwork(nn.Module):
    """
    Maps state → scalar value estimate V(s).
    Input:  (state_dim,)
    Output: scalar ()
    """
    def __init__(self, state_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            layer_init(nn.Linear(state_dim, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, hidden)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden, 1), std=1.0),  # std=1 for value head
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)   # (...,)


---
## Part A — VPG + Value Baseline on LunarLander

We re-run the Unit 1 algorithm unchanged: per-episode updates, MC returns.
This establishes what VPG achieves on a harder environment and makes the
comparison with A2C+GAE clean — the *only* thing we change in Part B is the
advantage estimator and the rollout strategy.


In [ ]:
def compute_mc_returns(rewards: list, gamma: float) -> torch.Tensor:
    """Discounted reward-to-go (Unit 1, O(T) with reverse trick)."""
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return torch.tensor(returns, dtype=torch.float32, device=device)  # (T,)


def vpg_with_baseline(
    env, actor, critic, actor_opt, critic_opt,
    n_episodes: int  = 600,
    gamma: float     = 0.995,
    print_every: int = 100,
) -> list:
    """
    VPG + value baseline — Algorithm 1 from Unit 1, unchanged.
    Advantage = R_t − V(s_t)  (full Monte Carlo return, per episode).
    """
    episode_rewards = []

    for ep in range(1, n_episodes + 1):
        state, _ = env.reset()
        states_ep, log_probs_ep, rewards_ep = [], [], []
        done = False

        while not done:
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            action, log_prob, _ = actor.get_action(state_t)
            state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            states_ep.append(state_t)
            log_probs_ep.append(log_prob)
            rewards_ep.append(reward)

        # ── Monte Carlo returns ───────────────────────────────────────────────
        returns   = compute_mc_returns(rewards_ep, gamma)   # (T,)

        # ── Critic forward ────────────────────────────────────────────────────
        states_t  = torch.stack(states_ep)                  # (T, 8)
        values    = critic(states_t)                         # (T,)

        # ── Advantages  Â_t = R_t − V(s_t) ──────────────────────────────────
        advantages = returns - values.detach()              # (T,)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # ── Actor loss ────────────────────────────────────────────────────────
        actor_loss  = -(torch.stack(log_probs_ep) * advantages).mean()

        # ── Critic loss ───────────────────────────────────────────────────────
        critic_loss = 0.5 * F.mse_loss(values, returns.detach())

        # ── Update ────────────────────────────────────────────────────────────
        actor_opt.zero_grad();  critic_opt.zero_grad()
        (actor_loss + critic_loss).backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), 0.5)
        actor_opt.step();  critic_opt.step()

        episode_rewards.append(sum(rewards_ep))

        if ep % print_every == 0:
            avg = np.mean(episode_rewards[-print_every:])
            print(f"[VPG]  ep {ep:4d}/{n_episodes}  avg: {avg:7.1f}")

    return episode_rewards


---
## Part B — A2C + GAE

The changes from Part A:

1. **N-step rollouts** (`N_STEPS=5`) instead of full episodes
2. **Parallel environments** (`N_ENVS=4`) — each step collects from 4 envs
3. **GAE(λ=0.95)** instead of MC returns — controlled bias/variance
4. **Episode-boundary masking** — `(1 - done)` prevents GAE from bleeding
   across episode boundaries

Everything else is identical: same network architecture, same optimizer,
same gradient clipping.


In [ ]:
def a2c_gae(
    actor, critic, actor_opt, critic_opt,
    n_envs:        int   = 4,        # parallel environments
    n_steps:       int   = 5,        # rollout length per update
    total_steps:   int   = 400_000,  # total environment steps
    gamma:         float = 0.995,    # discount — higher than CartPole (longer episodes)
    lam:           float = 0.95,     # GAE lambda
    entropy_coef:  float = 1e-4,     # small entropy bonus (from SB3-Zoo tuning)
    value_coef:    float = 0.5,
    max_grad_norm: float = 0.5,
    print_every:   int   = 50_000,   # print every N environment steps
    seed:          int   = 42,
) -> list:
    """
    A2C with GAE — N-step synchronous actor-critic.

    Per update (every n_steps × n_envs environment steps):
      1. Collect n_steps from each of n_envs parallel environments
      2. Compute V(s_t) for all collected states
      3. Compute GAE advantages with terminal masking
      4. Actor loss: −E[log π(a|s) · Â] − β·H(π)
      5. Critic loss: 0.5 · MSE(V(s), R_t)
      6. Combined backward + grad clip + step

    Hyperparameters match the rl-baselines3-zoo tuned config for
    LunarLander-v3 (https://github.com/DLR-RM/rl-baselines3-zoo).
    """
    # ── Setup parallel environments ───────────────────────────────────────────
    envs = gym.vector.SyncVectorEnv(
        [lambda: gym.make("LunarLander-v3") for _ in range(n_envs)]
    )
    obs, _ = envs.reset(seed=seed)
    obs    = torch.as_tensor(obs, dtype=torch.float32, device=device)
    # obs: (N_envs, 8)

    # Manual episode reward tracking (gymnasium SyncVectorEnv doesn't
    # auto-report episode statistics in v1.x without a wrapper)
    ep_buf     = np.zeros(n_envs)   # accumulated reward per live env
    all_ep_rew = []                 # completed episode totals

    n_updates = total_steps // (n_steps * n_envs)

    for update in range(n_updates):

        # ── 1. Allocate rollout buffers ───────────────────────────────────────
        # Shape: (n_steps, n_envs, ...) — time × env × feature
        obs_buf  = torch.zeros(n_steps, n_envs, STATE_DIM,  device=device)
        act_buf  = torch.zeros(n_steps, n_envs,             dtype=torch.long, device=device)
        logp_buf = torch.zeros(n_steps, n_envs,             device=device)
        rew_buf  = torch.zeros(n_steps, n_envs,             device=device)
        done_buf = torch.zeros(n_steps, n_envs,             device=device)
        val_buf  = torch.zeros(n_steps, n_envs,             device=device)

        # ── 2. Collect n_steps ────────────────────────────────────────────────
        for step in range(n_steps):
            with torch.no_grad():
                action, log_prob, _ = actor.get_action(obs)
                # action:   (N_envs,)
                # log_prob: (N_envs,)
                value = critic(obs)                                # (N_envs,)

            # Step all envs in parallel
            next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
            done = (terminated | truncated).astype(np.float32)

            # Track episode totals
            ep_buf += reward
            for i, d in enumerate(done):
                if d:
                    all_ep_rew.append(float(ep_buf[i]))
                    ep_buf[i] = 0.0

            obs_buf[step]  = obs
            act_buf[step]  = action
            logp_buf[step] = log_prob
            rew_buf[step]  = torch.as_tensor(reward, dtype=torch.float32, device=device)
            done_buf[step] = torch.as_tensor(done,   dtype=torch.float32, device=device)
            val_buf[step]  = value

            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

        # ── 3. Bootstrap: V(s_{T+1}) for the last observation ────────────────
        with torch.no_grad():
            next_val = critic(obs)    # (N_envs,)

        # ── 4. Compute GAE ────────────────────────────────────────────────────
        advantages, returns = compute_gae(
            rew_buf, val_buf.detach(), done_buf, next_val, gamma, lam
        )
        # advantages: (n_steps*n_envs,) — normalised
        # returns:    (n_steps*n_envs,) — critic regression targets

        # ── 5. Flatten buffers for the update ─────────────────────────────────
        obs_flat = obs_buf.reshape(-1, STATE_DIM)      # (T*N, 8)
        act_flat = act_buf.reshape(-1)                  # (T*N,)

        # ── 6. Forward pass (re-evaluate actions to get fresh log_probs) ──────
        _, new_log_probs, entropy = actor.get_action(obs_flat, act_flat)
        new_values                = critic(obs_flat)
        # new_log_probs: (T*N,)
        # entropy:       (T*N,)
        # new_values:    (T*N,)

        # ── 7. Losses ─────────────────────────────────────────────────────────
        actor_loss  = -(new_log_probs * advantages).mean()
        critic_loss = value_coef  * F.mse_loss(new_values, returns.detach())
        entropy_loss = -entropy_coef * entropy.mean()

        total_loss  = actor_loss + critic_loss + entropy_loss

        # ── 8. Update ─────────────────────────────────────────────────────────
        actor_opt.zero_grad();  critic_opt.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), max_grad_norm)
        actor_opt.step();  critic_opt.step()

        steps_done = (update + 1) * n_steps * n_envs
        if steps_done % print_every == 0 and all_ep_rew:
            avg = np.mean(all_ep_rew[-100:]) if len(all_ep_rew) >= 100                   else np.mean(all_ep_rew)
            print(f"[A2C+GAE]  step {steps_done:7d}  "
                  f"eps={len(all_ep_rew):4d}  avg100: {avg:7.1f}")

    envs.close()
    return all_ep_rew


---
## 7  Train — Same Seed, Same Architecture

We fix the random seed and use identical network architectures for both
parts.  The only differences: advantage estimator and rollout strategy.

**Expected runtime on Colab CPU:** ~10–15 minutes.
Note: for this network size, CPU is typically faster than a free Colab GPU
because the GPU overhead (memory transfers, kernel launches) exceeds the
tiny matrix-multiply time.  This is expected and not a `.to(device)` bug —
GPU wins when batch sizes and model sizes are large.


In [ ]:
SEED       = 42
HIDDEN_DIM = 256
LR         = 3e-4    # slightly higher than SB3 default (7e-4) for Adam vs RMSprop

# Hyperparameters from rl-baselines3-zoo tuned config for LunarLander-v3:
# github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/a2c.yml
GAMMA      = 0.995   # higher than CartPole's 0.99 (longer episodes need more foresight)
N_ENVS     = 4
N_STEPS    = 5
TOTAL_STEPS = 400_000

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
# ── Part A: VPG ───────────────────────────────────────────────────────────────
torch.manual_seed(SEED); np.random.seed(SEED)
env_a    = gym.make("LunarLander-v3")
env_a.reset(seed=SEED)
actor_a  = ActorNetwork(STATE_DIM, ACTION_DIM, HIDDEN_DIM).to(device)
critic_a = CriticNetwork(STATE_DIM, HIDDEN_DIM).to(device)
opt_a_actor  = optim.Adam(actor_a.parameters(),  lr=LR, eps=1e-5)
opt_a_critic = optim.Adam(critic_a.parameters(), lr=LR, eps=1e-5)

print("=" * 60)
print("Part A — VPG + value baseline  (per-episode MC returns)")
print("=" * 60)
rewards_vpg = vpg_with_baseline(
    env_a, actor_a, critic_a, opt_a_actor, opt_a_critic,
    n_episodes=600, gamma=GAMMA,
)
env_a.close()
print(f"\nVPG final avg (last 100 eps): {np.mean(rewards_vpg[-100:]):.1f}")


In [ ]:
# ── Part B: A2C + GAE ─────────────────────────────────────────────────────────
torch.manual_seed(SEED); np.random.seed(SEED)
actor_b  = ActorNetwork(STATE_DIM, ACTION_DIM, HIDDEN_DIM).to(device)
critic_b = CriticNetwork(STATE_DIM, HIDDEN_DIM).to(device)
opt_b_actor  = optim.Adam(actor_b.parameters(),  lr=LR, eps=1e-5)
opt_b_critic = optim.Adam(critic_b.parameters(), lr=LR, eps=1e-5)

print("=" * 60)
print(f"Part B — A2C + GAE  (N={N_STEPS} steps · {N_ENVS} envs · λ=0.95 · γ={GAMMA})")
print("=" * 60)
rewards_a2c = a2c_gae(
    actor_b, critic_b, opt_b_actor, opt_b_critic,
    n_envs=N_ENVS, n_steps=N_STEPS, total_steps=TOTAL_STEPS,
    gamma=GAMMA, lam=0.95, seed=SEED,
)
print(f"\nA2C+GAE final avg (last 100 eps): {np.mean(rewards_a2c[-100:]):.1f}")


---
## 8  Comparison — VPG vs A2C + GAE


In [ ]:
def smooth(values: list, window: int = 50) -> list:
    out = []
    for i in range(len(values)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(values[lo : i + 1])))
    return out


fig, ax = plt.subplots(figsize=(12, 5))

# VPG
eps_vpg = range(1, len(rewards_vpg) + 1)
ax.plot(eps_vpg, rewards_vpg, alpha=0.12, color="steelblue", linewidth=0.7)
ax.plot(eps_vpg, smooth(rewards_vpg, 50), color="steelblue", linewidth=2.5,
        label="VPG + MC returns (per-episode)")

# A2C+GAE
eps_a2c = range(1, len(rewards_a2c) + 1)
ax.plot(eps_a2c, rewards_a2c, alpha=0.12, color="darkorange", linewidth=0.7)
ax.plot(eps_a2c, smooth(rewards_a2c, 50), color="darkorange", linewidth=2.5,
        label="A2C + GAE (λ=0.95)")

ax.axhline(200, color="red", linestyle="--", linewidth=1.5,
           alpha=0.8, label="Solved threshold (200)")

ax.set_xlabel("Episode", fontsize=12)
ax.set_ylabel("Total Reward", fontsize=12)
ax.set_title("VPG vs A2C+GAE on LunarLander-v3", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("comparison.png", dpi=130)
plt.show()
print("Saved → comparison.png")


### Interpreting the results

#### What to look for

Exact numbers depend on seed and hardware, but the qualitative story is robust:

**VPG (blue)** — per-episode MC advantage:
- Climbs slowly from ~$-200$ initially
- Shows large oscillations / sharp collapses (the value-bootstrap trap
  from Unit 1 is even more pronounced on LunarLander's long episodes)
- Plateaus well below the 200 threshold

**A2C + GAE (orange)** — N-step with terminal masking:
- Converges more steadily and reaches higher rewards
- Much lower amplitude oscillations — the N-step rollout prevents a
  single long bad episode from dominating the gradient
- Gets closer to (or crosses) the 200 threshold

#### Why A2C+GAE is more stable

1. **N-step rollouts limit the gradient scale.** Each update uses exactly
   $N \times N_e = 20$ transitions.  With per-episode VPG, one 1000-step
   crash contributes 200× more to the loss than a 5-step episode.

2. **Terminal masking prevents GAE bleed.** When a LunarLander episode ends
   mid-rollout (e.g. step 3 of a 5-step window), the $(1-\text{done})$ factor
   zeros out the bootstrap for subsequent steps.  Without this, the value of
   the *new* episode leaks into the advantage estimate of the *old* one.

3. **$\lambda=0.95$ uses ~20 effective steps** instead of the full 300–1000.
   The signal *"moved toward the pad"* arrives in a few steps, not after a
   crash 800 steps later.

#### Why LunarLander is still hard for A2C

Even with all these improvements, A2C alone (one gradient step per rollout)
often doesn't fully solve LunarLander within a reasonable compute budget.
This is consistent with published results: the
[rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo) pre-trained
A2C model for LunarLander-v3 averages only ~32 reward over 200k steps.

The remaining problem: **there is still no limit on the step size.**
A2C+GAE improves the advantage *estimate*, but nothing stops a single large
gradient from overwriting a good policy.

This is exactly what **PPO** (Unit 3) fixes.


In [ ]:
def evaluate_greedy(actor: ActorNetwork, n_episodes: int = 50) -> float:
    """Run n_episodes greedy (argmax logits) and return mean total reward."""
    eval_env = gym.make("LunarLander-v3")
    total_rewards = []
    actor.eval()
    with torch.no_grad():
        for _ in range(n_episodes):
            state, _ = eval_env.reset()
            total, done = 0.0, False
            while not done:
                s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
                action = actor(s_t).argmax().item()   # greedy: argmax logits
                state, r, terminated, truncated, _ = eval_env.step(action)
                done = terminated or truncated
                total += r
            total_rewards.append(total)
    eval_env.close()
    actor.train()
    return float(np.mean(total_rewards))


score_vpg = evaluate_greedy(actor_a)
score_a2c = evaluate_greedy(actor_b)

print("Greedy evaluation (50 episodes, argmax policy):")
print(f"  VPG + MC returns : {score_vpg:6.1f}  {'✓ SOLVED' if score_vpg >= 200 else 'not solved'}")
print(f"  A2C + GAE        : {score_a2c:6.1f}  {'✓ SOLVED' if score_a2c >= 200 else 'not solved'}")


---
## 9  Watch the A2C Agent


In [ ]:
def record_gif(actor: ActorNetwork, filename: str = "agent.gif",
               max_steps: int = 1000) -> str:
    env_rec = gym.make("LunarLander-v3", render_mode="rgb_array")
    frames, total_reward = [], 0.0
    state, _ = env_rec.reset()
    done = False
    actor.eval()
    with torch.no_grad():
        while not done and len(frames) < max_steps:
            frames.append(env_rec.render())
            s_t    = torch.as_tensor(state, dtype=torch.float32, device=device)
            action = actor(s_t).argmax().item()
            state, r, terminated, truncated, _ = env_rec.step(action)
            done = terminated or truncated
            total_reward += r
    env_rec.close()
    actor.train()
    iio.imwrite(filename, np.stack(frames), plugin="pillow", loop=0, duration=33)
    print(f"{len(frames)} frames  |  total reward: {total_reward:.1f}  →  {filename}")
    return filename


gif_path = record_gif(actor_b)
display(Image(gif_path))


---
## 10  What's Next — PPO

A2C+GAE gives us a much better advantage estimate, but the policy can still
be destroyed by a single large gradient step.  The fundamental issue:
**one rollout → one unconstrained update**.

**PPO** (Proximal Policy Optimisation, Schulman et al. 2017) solves this
with two ideas:

1. **Reuse each rollout for multiple gradient steps** (typically 4–10 epochs)
   so we squeeze more learning out of each batch of experience.

2. **Clip the probability ratio** to prevent any single update from moving
   the policy too far:

$$L^{\text{CLIP}} = \mathbb{E}_t \left[
  \min\!\left(
    r_t(\theta)\,\hat{A}_t,\;
    \text{clip}\bigl(r_t(\theta),\, 1-\varepsilon,\, 1+\varepsilon\bigr)\,\hat{A}_t
  \right)
\right]$$

where $r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$
is the ratio between new and old policy probability.

When the policy tries to move too far ($r_t$ deviates from 1), the clip
kills the gradient for that sample.  **Catastrophic collapse becomes
impossible by construction.**

PPO uses exactly the same GAE advantage estimator as A2C — the only
difference is the clipped loss and the multi-epoch update.  On LunarLander,
PPO reliably crosses the 200 threshold in the same compute budget where
A2C struggles.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
